This notebook is intended for a short demonstration of how can wastewater data be used to analyse dynamics of a viral pathogen. It is meant as an illustration of some principles, using mainly base R functions rather than dedicated software, with the goal of giving some intuition. It is not meant as a comprehensive tutorial on wastewater data analysis, and it is not meant to be used for actual wastewater data analysis!

In this notebook, we will use data produced by the [WISE consortium](https://wise.ethz.ch/)

# Part 1: tracking viral loads
## Overview
We will do some exploratory wastewater analysis using SARS-CoV-2 concentrations data from Zürich, Switzerland.
Data are publicly provided by EAWAG (Swiss Federal Institute of Aquatic Science and Technology) under a CC BY 4.0 license. We will compare them with daily new cases communicated by the FOPH (Federal Office of Public Healh. We are loading them here via the [EpiSewer](https://github.com/adrian-lison/EpiSewer) repository.


In [ ]:
# install.packages("robustbase")
install.packages("reshape2")
library(ggplot2)
theme_set(theme_minimal())
options(repr.plot.width=10, repr.plot.height=5)
# make defaults for larger plot 

In [ ]:
data_url <- "https://raw.githubusercontent.com/adrian-lison/EpiSewer/main/data/SARS_CoV_2_Zurich.rda"
tmp_file <- tempfile(fileext = ".rda")
download.file(data_url, tmp_file, mode = "wb", quiet = TRUE)

obj_env <- new.env()
loaded_names <- load(tmp_file, envir = obj_env)
data_zurich <- obj_env[[loaded_names[1]]]

names(data_zurich)

## Initial data checks
We first inspect the main time series in the dataset: reported cases and wastewater concentration. 

In [ ]:
# Reported cases over time
ggplot() +
  geom_step(aes(x = data_zurich$cases$date, y = data_zurich$cases$cases, color = "Cases")) +
  labs(title = "Reported Cases Over Time", x = "Date", y = "Cases", color = NULL)

# Wastewater concentration over time
ggplot() +
  geom_point(aes(x = data_zurich$measurements$date, y = data_zurich$measurements$concentration, color = "Concentration"), alpha = 0.7) +
  geom_line(aes(x = data_zurich$measurements$date, y = data_zurich$measurements$concentration, color = "Concentration"), linewidth = 0.4) +
  labs(title = "Wastewater Concentration Over Time", x = "Date", y = "Concentration", color = NULL)


Next, we merge cases, flow, and concentration by date so that all variables are aligned in a single table for downstream analysis.

In [ ]:
# Merge by date to create one analysis table
merged_data <- merge(data_zurich$cases, data_zurich$flows, by = "date", all = FALSE)
merged_data <- merge(merged_data, data_zurich$measurements, by = "date", all = FALSE)

## Normalisation for flow 

Daily observed concentrations can be affected by external factors such as rainfall. To try and mitigate this, we will normalise using flow data (daily inflow at the treatment plant).

In [ ]:
# plot flows
ggplot() +
  geom_point(aes(x = merged_data$date, y = merged_data$flow, color = "Flow"), alpha = 0.7) +
  geom_line(aes(x = merged_data$date, y = merged_data$flow, color = "Flow"), linewidth = 0.4) +
  labs(title = "Wastewater Flow Over Time", x = "Date", y = "Flow", color = NULL)

Check the units:

In [ ]:
data_zurich$units

We will compute wastewater loads (gc/day) = concentration (gc/mL) * flow (mL/day)

In [ ]:
# Approximate wastewater load
merged_data$loads <- merged_data$concentration * merged_data$flow #/ median(merged_data$flow, na.rm = TRUE)

## Relate wastewater loads to reported cases

We will visualise first the loads and cases on the same plot, and align them to get a sense of their relationship. 

In [ ]:
# We plot loads and cases together to get a sense of their relationship
# The scales are different, we will add a second axis on the right for loads
# to do this we need to rescale loads to be on a similar scale to cases, for visualisation only

rescale_factor <- median(merged_data$cases, na.rm = TRUE) / median(merged_data$loads, na.rm = TRUE)
merged_data$loads_rescaled <- merged_data$loads * rescale_factor
ggplot() +
  geom_step(aes(x = merged_data$date, y = merged_data$cases, color = "Cases")) +
  geom_point(aes(x = merged_data$date, y = merged_data$loads_rescaled, color = "Loads (rescaled)"), alpha = 0.7) +
  geom_line(aes(x = merged_data$date, y = merged_data$loads_rescaled, color = "Loads (rescaled)"), linewidth = 0.4) +
  scale_y_continuous(
    name = "Cases",
    sec.axis = sec_axis(~ . / rescale_factor, name = "Loads")
  ) +
  labs(title = "Reported Cases and Wastewater Loads Over Time", x = "Date", color = NULL)

We see that there seem to be a good correspondence between the two time series. However, it seems that the ratio of loads to cases changes between the two waves of cases (why?). Let's cut the data in two periods and fit a simple regression model to each period to quantify the relationship between loads and cases.

In [ ]:
# Split date to compare periods
merged_data$cutoff <- ifelse(merged_data$date <= as.Date("2022-02-15"),yes = "before", no = "after")

# Linear-scale comparison
ggplot(data = merged_data) +
  geom_point(aes(x = cases, y = loads, color = cutoff), alpha = 0.7) +
  geom_smooth(aes(x = cases, y = loads, color = cutoff),
#   method = robustbase::lmrob,
   method = lm) +
  labs(title = "Cases vs Wastewater Loads", x = "Reported cases", y = "Wastewater Loads", color = "cutoff")


We fit a linear model to summarize the association between wastewater load and reported cases, allowing for a change before and after the cutoff date.

In [ ]:
# model summary: loads as a function of cases and period
# summary(robustbase::lmrob(loads ~ cases * cutoff, data = merged_data))
summary(lm(loads ~ cases * cutoff, data = merged_data))

## Epidemiological dynamics
Both types of data are noisy (though the noise seem to be quite different). We apply a simple kernel smoother to both case counts and wastewater loads to visualize broader temporal patterns.

In [ ]:
# Smooth wastewater loads over time
ok <- is.finite(merged_data$date) & is.finite(merged_data$loads) # only select rows where both date and loads are not NA
merged_data$loads_smooth <- ksmooth(
  x = merged_data$date[ok],
  y = merged_data$loads[ok],
  kernel = "normal",
  bandwidth = 14,
  x.points = merged_data$date
)$y

# Smooth reported cases over time
ok <- is.finite(merged_data$date) & is.finite(merged_data$cases)
merged_data$cases_smooth <- ksmooth(
  x = merged_data$date[ok],
  y = merged_data$cases[ok], 
  kernel = "normal",
  bandwidth = 14,
  x.points = merged_data$date
)$y

# Plot smoothed loads and cases together
ggplot() +
  geom_line(aes(x = merged_data$date, y = merged_data$cases_smooth, color = "Cases (smoothed)"), linewidth = 0.8) +
  geom_line(aes(x = merged_data$date, y = merged_data$loads_smooth * rescale_factor, color = "Loads (smoothed)"), linewidth = 0.8) +
  geom_step(aes(x = merged_data$date, y = merged_data$cases, color = "Cases (smoothed)"), alpha = 0.5) +
  geom_point(aes(x = merged_data$date, y = merged_data$loads * rescale_factor, color = "Loads (smoothed)"), alpha = 0.5) +
  scale_y_continuous(
    name = "Cases (smoothed)",
    sec.axis = sec_axis(~ . / rescale_factor, name = "Loads (smoothed)")
  ) +
  labs(title = "Smoothed Reported Cases and Wastewater Loads Over Time", x = "Date", color = NULL)  



## Effective reproduction number

The effective reproduction number $R(t)$ is a key epidemiological metric that quantifies the average number of secondary infections generated by an infected individual at time $t$. It provides insights into the transmission dynamics of an infectious disease and helps to assess whether an outbreak is growing ($R(t) > 1$), shrinking ($R(t) < 1$), or stable ($R(t) = 1$).

In a compartmental model, one way to relate the effective reproduction number to the time-varying growth rate $r(t)$ of the number of infected individuals $I(t)$ is as follows:

$$
R(t) = 1 + g r(t)
$$

where $g$ is the average generation time. Here we use $g = 5$ days as a simple assumption. As $r(t)$ can be estimated by

$$
\frac{d I(t)}{dt} = r(t) I(t) \implies r(t) = \frac{1}{I(t)} \frac{d I(t)}{dt} = \frac{d\log(I(t))}{dt}
$$

We can estimate the effective reproduction number from the smoothed trajectories using:

$$
R(t) = 1 + g\frac{d\log(I(t))}{dt}
$$



In [ ]:
# Approximate R from smoothed loads and smoothed cases
merged_data$R_loads <- c(NA, 1 + 5 * diff(log(merged_data$loads_smooth)))
merged_data$R_cases <- c(NA, 1 + 5 * diff(log(merged_data$cases_smooth)))

# Compare both R proxies over time
ggplot(data = merged_data) +
  geom_line(aes(x = date, y = R_loads, color = "R from loads"), linewidth = 0.7) +
  geom_line(aes(x = date, y = R_cases, color = "R from cases"), linewidth = 0.7) +
  geom_hline(yintercept = 1, linetype = "dashed") +
  labs(title = "Estimated Effective Reproduction Number", x = "Date", y = "R(t)", color = NULL)

We see that despite the wastewater loads and confirmed cases time series having different scales and characteristics, the reproduction number estimates coincide well. 

In practice, one would use more sophisticated methods to estimate the effective reproduction number, and they can be applied to both case counts and wastewater data. Those typically adjust for shedding loads profiles, testing delay distribution, and model the noise more explicitly to provide measures of uncertainty. 

To go further, see [EpiSewer](https://github.com/adrian-lison/EpiSewer). 

# Part 2: sequencing data
## Overview
In this part, we will add information on viral variants derived from sequencing data. Having information on the circulating variant will add more context to the observed trends in loads and cases, and can help us understand how different variants may have contributed to the observed dynamics. 

We will use data from [this study on modeling viral variants competition from wastewater sequencing data](https://www.medrxiv.org/content/10.1101/2025.03.25.25324639v1)

Let's load the mutation data. Here, we provide wastewater sequencing data which has already been processed an mutations called on a set of positions of interest. 

In [ ]:
mut_calls <- read.csv(
    "https://github.com/dr-david/tutorial_cambodia_2026/blob/main/data/wastewater_mut_calls.csv?raw=true",
    stringsAsFactors = FALSE, check.names = FALSE
    )
# mut_calls <- read.csv("data/wastewater_mut_calls.csv")
mut_calls$date <- as.Date(mut_calls$date)

ggplot(mut_calls) + 
    geom_tile(aes(x=date, y=factor(pos), fill=frac)) + 
    labs(
        title = "Fraction of mutated reads over time",
        x = "Date",
        y = "Mutation position",
        fill = "Fraction"
    )

These mutation frequencies are explained by the presence in the samples of multiple variants. Each variant has a specific set of mutations, and the observed mutation frequencies are a mixture of the frequencies of the different variants. If we denote $v_t$ as the frequency vector of variants in the sample at time $t$, $m_t$ as the vector of mutation frequencies at time $t$, then the relationship between variant frequencies and mutation frequencies can be expressed as:

$$
m_t \approx M v_t
$$

Where $M$ is a binary matrix that indicates which mutations are present in each variant. We can visualise this matrix:

In [ ]:
mut_def <- mut_calls[!duplicated(mut_calls$pos), ]
mut_def <- mut_def[order(as.numeric(mut_def$pos)), ]

variant_cols <- c("B.1.617.2", "BA.1", "BA.2")

# Long format: one row per (pos, variant) with mutation presence in `present`
mut_def_long <- reshape2::melt(
  mut_def[, c("pos", variant_cols)],
  id.vars = "pos",
  measure.vars = variant_cols,
  variable.name = "variant",
  value.name = "present"
)

mut_def_long$present <- as.numeric(mut_def_long$present)
# mut_def_long

ggplot(mut_def_long) +
  geom_tile(aes(x = variant, y = factor(pos), fill = factor(present))) +
  labs(title = "Mutation presence by variant", x = "Variant", y = "Position", fill = "Present") 

We see that the variants BA.1 and BA.2 share many mutations, but also have some distinct ones. 

We will load the sequencing data already deconvolved into the relative abundace of the different variants:

In [ ]:
# Load deconvolved variant proportions 
deconvolved <- read.csv("https://github.com/dr-david/tutorial_cambodia_2026/blob/main/data/deconvolved_zurich.csv?raw=true", stringsAsFactors = FALSE, check.names = FALSE)
# deconvolved <- read.csv("./deconvolved_zurich.csv",  stringsAsFactors = FALSE, check.names = FALSE)
deconvolved$date <- as.Date(deconvolved$date)

ggplot(data=deconvolved) +
 geom_line(aes(x=date, y=proportion, color=variant)) + 
 labs(
    title = "Relative abundance of variants in Zurich wastewater",
    x = "Date",
    y = "fraction",
    fill = NULL
  )



We can merge the wastewater loads with the wastewater sequencing data:

In [ ]:

# Merge with loads
loads_by_variant <- merge(
  deconvolved,
  merged_data[, c("date", "loads")],
  by = "date",
  all.x = TRUE
)

# Variant-attributed loads (proportional split)
loads_by_variant$loads_variant <- loads_by_variant$loads * loads_by_variant$proportion


# Plot loads colored by variant fraction
ggplot(data = loads_by_variant) +
  geom_area(aes(x = date, y = loads_variant, fill = variant), position = "stack") +
  labs(
    title = "Variant-Stratified Wastewater Loads in Zurich",
    x = "Date",
    y = "Loads",
    fill = NULL
  )


We see that the second wave in the datset corresponds to the emergence and spread of the BA.2 variant. 

## Selection 

If a variant has a higher fitness (tied to an increase in its reproduction number) than another, it will tend to increase in frequency over time. We can in turn use the observed variant frequencies to estimate the relative selective advantage of the variants.

One simple way to derive this: if we assume that variants $X$ and $Y$ are growing exponentially with growth rates $r_X$ and $r_Y$, then their fraction $f_X$ and $f_Y$ will change over time according to:  
$$
f_X(t) = \frac{X(0)e^{r_X t}}{X(0)e^{r_X t} + Y(0)e^{r_Y t}} \quad\text{and}\quad f_Y(t) = \frac{Y(0)e^{r_Y t}}{X(0)e^{r_X t} + Y(0)e^{r_Y t}}
$$

Substituting $r_X = r_Y + s$ (where $s$ is the selection coefficient) gives:
$$
f_X(t) = \frac{X(0)e^{(r_Y + s) t}}{X(0)e^{(r_Y + s) t} + Y(0)e^{r_Y t}} = \frac{X(0)e^{s t}}{X(0)e^{s t} + Y(0)}  = \frac{1}{1 + \frac{Y(0)}{X(0)}e^{-s t}} = \text{logit}^{-1}\left(s t + \log\frac{X(0)}{Y(0)}\right)
$$

We can therefore estimate the selection coefficient $s$ by fitting a logistic regression model to the observed variant frequencies over time. The rate parameter of the logistic regression will give us an estimate of the selection coefficient $s$. 

In [ ]:
# We will first prepare the data by subsetting just the BA.2 variant, and removing dates where "undetermined" is too high

# find dates where undetermined is above 0.5
undetermined_threshold <- 0.5
undetermined_high_dates <- loads_by_variant$date[loads_by_variant$variant == "undetermined" & loads_by_variant$proportion > undetermined_threshold] 

# subset BA.2 data, excluding dates where undetermined is high
ba2_dataset <- loads_by_variant[loads_by_variant$variant == "BA.2" & !loads_by_variant$date %in% undetermined_high_dates, ]


ggplot(ba2_dataset) +
  geom_line(aes(x = date, y = proportion)) +
  labs(
    title = "Fraction of BA.2 variant in wastewater over time",
    x = "Date",
    y = "BA.2 relative fraction"
  )

We can fit a logistic regression GLM to this data to estimate $s$

In [ ]:
fit_ba2 <- glm(proportion ~ date, data = ba2_dataset, family = quasibinomial(link = "logit"))

ggplot(ba2_dataset) +
  geom_line(aes(x = date, y = proportion, color="observed")) +
  geom_line(aes(x = date, y = predict(fit_ba2, type = "response"), color="fit")) +
    labs(
        title = "Fraction of BA.2 variant in wastewater over time",
        x = "Date",
        y = "BA.2 relative fraction"
    )

We see that despite its simplicity, this model fits the data quite well. 

We can translate the estimated selection coefficient into an estimate of the relative reproduction number of the variant using the formula:
$$R_X = R_Y e^{s g} \Rightarrow R_X / R_Y = e^{s g}
$$
where $g$ is the expected generation time. We assume here $g = 5$ days. 

In [ ]:
s <- unname(coef(fit_ba2)[2]) # extract coef
ci <- confint(fit_ba2)[2, ] # extract confidence intervals

rr <- exp(c(est = s, lo = ci[1], hi = ci[2]) * 5) # convert into relative R advantage 


cat(
    sprintf("Relative R advantage and .95 confint: %.1f%%  [%.1f%%, %.1f%%]\n",
                    100 * rr[1], 100 * rr[2], 100 * rr[3])
)


## Forecasting

We can use this simple model for forecasting the progression of the BA.2 variant. 

We will use observations only for the first couple weeks of data, right after the introduction of the new variant, fit the model to them, and forecast the rest of the spread:

In [ ]:
# Refit BA.2 fraction model using data only up to 15 January, then predict forward

variant_cutoff_date <- as.Date("2022-01-15")
ba2_short <- ba2_dataset[ba2_dataset$date <= variant_cutoff_date, ]

# refit the model on the short dataset
fit_ba2_short <- glm(proportion ~ date, data = ba2_short, family = quasibinomial(link = "logit"))

# obtain fitted values and confidence intervals on the logit scale
predicted_short_link <- predict(fit_ba2_short, newdata = ba2_dataset, type = "link", se.fit = TRUE)

# transform predictions and intervals back to fraction scale
predicted_short <- plogis(predicted_short_link$fit)
predicted_short_ci <- data.frame(
  date = ba2_dataset$date,
  fit = predicted_short,
  lower = plogis(predicted_short_link$fit - 1.96 * predicted_short_link$se.fit),
  upper = plogis(predicted_short_link$fit + 1.96 * predicted_short_link$se.fit)
)

ggplot(ba2_dataset) +
  geom_line(aes(x = date, y = proportion, color = "observed data")) +
  geom_line(aes(x = date, y = predicted_short, color = "fit + prediction"), linetype = "dashed") +
  geom_ribbon(
    data = predicted_short_ci,
    aes(x = date, ymin = lower, ymax = upper),
    fill = "red", alpha = 0.2, inherit.aes = FALSE
  ) +
  geom_vline(xintercept = variant_cutoff_date, linetype = "dashed", color = "black") +
  # add (observed / forecast) annotation to before and after the cutoff
  annotate("text", x = variant_cutoff_date - 8, y = 0.8, label = "fitted", color = "black") +
  annotate("text", x = variant_cutoff_date + 8, y = 0.8, label = "forecast", color = "black") +
  labs(
    title = "Fraction of BA.2 variant in wastewater over time",
    x = "Date",
    y = "BA.2 relative fraction"
  )

Despite a slight overestimation of the variant fitness, the forecast is quite good for limited data. 

## Compare with clinical sequencing 

We can load some count data of clinical sequencing from the same period, and compare the observed variant frequencies and fitted fitness model in the wastewater with those based on clinical samples. The clinical sequencing counts have been previously queried from [LAPIS](https://lapis.cov-spectrum.org/open/v2/docs/getting-started/introduction/)

In [ ]:
# Load data 
clinical_seq <- read.csv("https://github.com/dr-david/tutorial_cambodia_2026/blob/main/data/clinical_seq_zurich.csv?raw=true", stringsAsFactors = FALSE, check.names = FALSE)
# clinical_seq <- read.csv("clinical_seq_zurich.csv")
clinical_seq$date <- as.Date(clinical_seq$date)

# Fit model
fit_ba2_clinical <- glm(
    I(BA.2./total_count) ~ date,
    weights = clinical_seq$total_count, 
    data = clinical_seq, family = quasibinomial(link = "logit")
)

# plot
ggplot() + 
geom_step(data=clinical_seq, aes(x=date, y=BA.2./total_count, color="clinical"), stat="identity") + 
  geom_line(data = clinical_seq, aes(x = date, y = predict(fit_ba2_clinical, type = "response"), color="clinical")) + 
  geom_line(data = ba2_dataset, aes(x = date, y = proportion, color="wastewater")) +
  geom_line(data = ba2_dataset, aes(x = date, y = predict(fit_ba2, type = "response"), color="wastewater")) +
    labs(
        title = "Fraction of BA.2 variant in wastewater and clinical data over time",
        x = "Date",
        y = "BA.2 relative fraction"
    )




We see that the progression of the variant can be somewhat similarly tracked from the wastewater sequencing or clinical sequencing data. 

Let's see what happens if we try forecasting the spread of the variant from the first few weeks of observations:

In [ ]:
# fit clinical model on shorter data
fit_ba2_clinical_short <- glm(I(BA.2./total_count) ~ date,
 weights = clinical_seq[clinical_seq$date <= variant_cutoff_date,]$total_count, 
 data = clinical_seq[clinical_seq$date <= variant_cutoff_date,], family = quasibinomial(link = "logit"))

# obtain fitted values and confidence intervals on the logit scale
predicted_short_link_clinical <- predict(fit_ba2_clinical_short, newdata = clinical_seq, type = "link", se.fit = TRUE)

# transform predictions and intervals back to fraction scale
predicted_short_clinical <- plogis(predicted_short_link_clinical$fit)
predicted_short_ci_clinical <- data.frame(
  date = clinical_seq$date,
  fit = predicted_short_clinical,
  lower = plogis(predicted_short_link_clinical$fit - 1.96 * predicted_short_link_clinical$se.fit),
  upper = plogis(predicted_short_link_clinical$fit + 1.96 * predicted_short_link_clinical$se.fit)
)

ggplot(clinical_seq) +
  geom_line(aes(x = date, y = BA.2./total_count, color = "observed data")) +
  geom_line(aes(x = date, y = predicted_short_clinical, color = "fit + prediction"), linetype = "dashed") +
  geom_ribbon(
    data = predicted_short_ci_clinical,
    aes(x = date, ymin = lower, ymax = upper),
    fill = "red", alpha = 0.2, inherit.aes = FALSE
  ) +
  geom_vline(xintercept = variant_cutoff_date, linetype = "dashed", color = "black") +
  # add (observed / forecast) annotation to before and after the cutoff
  annotate("text", x = variant_cutoff_date - 8, y = 0.8, label = "fitted", color = "black") +
  annotate("text", x = variant_cutoff_date + 8, y = 0.8, label = "forecast", color = "black") +
  labs(
    title = "Fraction of BA.2 variant in clinical samples over time",
    x = "Date",
    y = "BA.2 relative fraction"
  )

We see though that, if we take the same dates cutoff for forecasting, we haven't yet observed enough clinical data to get a good estimate of the variant fitness, and therefore the forecast is not good. 

## Compare early estimation of fitness

We will assess how our estimates of the fitness advantage of the variant would have changed as new data became available, both with the wastewater and clinical data

In [ ]:
# we will fit the model on cutoffs from the start date to the end of the dataset
# fitting the model on wasteawter and clinical data, and recording the estimates

cutoffs <- seq(min(ba2_dataset$date), max(ba2_dataset$date), by = "1 day")

estimates_wastewater <- lapply(cutoffs, function(cutoff) {
  model <- glm(proportion ~ date, data = ba2_dataset[ba2_dataset$date <= cutoff, ], family = quasibinomial(link = "logit"))
  return(coef(model)[2])
})
estimates_wastewater <- unlist(estimates_wastewater)

estimates_clinical <- lapply(cutoffs, function(cutoff) {
  model <- glm(I(BA.2./total_count) ~ date,
 weights = clinical_seq[clinical_seq$date <= cutoff, ]$total_count, 
 data = clinical_seq[clinical_seq$date <= cutoff, ], family = quasibinomial(link = "logit"))
  return(coef(model)[2])
})
estimates_clinical <- unlist(estimates_clinical)

ggplot() +
  geom_line(aes(x = cutoffs, y = exp(5*estimates_wastewater) * 100, color = "wastewater")) +
  geom_line(aes(x = cutoffs, y = exp(5*estimates_clinical) * 100, color = "clinical")) +
  coord_cartesian(ylim = c(0, 400)) +
  labs(
    title = "Estimated selection advantage of BA.2 as more data is added",
    x = "Cutoff date for model fitting",
    y = "Estimated selection advantage",
    color = NULL
  )

# plot the cumulative number of samples as a function of time, as with geom_step
# for the clinical data, it is in the column total_count
# for the wasteawter data, each row is a sample, so we can just count the number of rows up to each date and plot
ggplot() +
  geom_step(data=clinical_seq, aes(x = date, y = cumsum(total_count), color="clinical")) +
  geom_step(data=ba2_dataset, aes(x = date, y = seq_along(date), color="wastewater")) +
  labs(
    title = "Cumulative number of sequenced samples over time",
    x = "Date",
    y = "# sequenced samples"
  ) + scale_y_log10()